# Organized research notebook

Cell source is retained from the supplied research archive. Saved outputs and credentials have been removed. Run the setup cell first. Check model, SAE, file paths, and run settings in the cells before starting a GPU experiment. This notebook has been checked statically, not rerun against the paper results. See `docs/RELEASE_REVIEW.md` for unresolved items.


In [ ]:
from pathlib import Path
import os
import sys

REPO_ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
                  if (p / "src" / "sae_emotion" / "__init__.py").exists()), None)
if REPO_ROOT is None:
    raise RuntimeError("Open this notebook from inside the sae-emotion repository.")
sys.path.insert(0, str(REPO_ROOT / "src"))
os.chdir(REPO_ROOT)


In [ ]:
import os
import pandas as pd
import numpy as np
from IPython.display import display

In [ ]:
# ============================================================
# Config
# ============================================================


MODEL_NAME = "google/gemma-2-2b"
model_slug = MODEL_NAME.replace("/", "-")

PAIR_OUT_DIR = (
    "data/google-gemma-2-2b-feature-act/"
    "adjacent_group_route_mediation/"
    "adjacent_group_path_patching"
)

PAIR_SUMMARY_PATH = os.path.join(
    PAIR_OUT_DIR,
    "adjacent_group_pair_summary.csv",
)

EMOTION_SUMMARY_PATH = os.path.join(
    PAIR_OUT_DIR,
    "adjacent_group_emotion_summary.csv",
)

TABLE_OUT_DIR = os.path.join(
    "data/google-gemma-2-2b-feature-act/",
    "adjacent_group_route_mediation",
    "tables",
)

os.makedirs(TABLE_OUT_DIR, exist_ok=True)

print("Pair summary:", PAIR_SUMMARY_PATH)
print("Emotion summary:", EMOTION_SUMMARY_PATH)
print("Table output:", TABLE_OUT_DIR)


# ============================================================
# Helpers
# ============================================================

def _fmt_float(x, digits=3):
    if pd.isna(x):
        return ""
    return round(float(x), digits)


def make_pair_result_table(
    pair_summary_path: str,
    output_dir: str,
    top_k_per_emotion: int | None = None,
    sort_by: str = "mean_mediation_recovery_ratio",
):
    df = pd.read_csv(pair_summary_path)

    # Recompute route support just in case.
    df["supports_route_pair"] = (
        (df["mean_m_mean_drop_after_x_ablation"] > 0)
        & (df["mean_x_to_y_logit_loss"] > 0)
        & (df["mean_m_to_y_logit_loss"] > 0)
        & (df["mean_mediation_restored_delta"] > 0)
    )

    # Compact paper table.
    keep_cols = [
        "target_emotion",
        "source_layer",
        "restore_layer",
        "n_source_features",
        "n_restore_features",

        # X -> M
        "mean_m_mean_drop_after_x_ablation",
        "rate_x_ablation_decreases_m_activation",

        # X -> Y
        "mean_x_to_y_logit_loss",
        "rate_x_ablation_lowers_logit",

        # M -> Y
        "mean_m_to_y_logit_loss",
        "rate_m_ablation_lowers_logit",

        # X -> M -> Y
        "mean_mediation_restored_delta",
        "mean_mediation_recovery_ratio",
        "rate_restore_increases_logit_after_x_ablation",

        # Overall evidence
        "rate_supports_route_instance",
        "supports_route_pair",

        # Prediction flip metrics
        "flip_rate_clean_to_x_ablate",
        "flip_rate_clean_to_m_ablate",
        "flip_rate_x_ablate_to_restore",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    table = df[keep_cols].copy()

    # Optional: keep strongest layer pairs per emotion.
    if top_k_per_emotion is not None:
        table = (
            table.sort_values(
                ["target_emotion", sort_by],
                ascending=[True, False],
            )
            .groupby("target_emotion", group_keys=False)
            .head(top_k_per_emotion)
            .reset_index(drop=True)
        )
    else:
        table = table.sort_values(
            ["target_emotion", "source_layer", "restore_layer"]
        ).reset_index(drop=True)

    # Rename columns for readability.
    rename_map = {
        "target_emotion": "Emotion",
        "source_layer": "Source layer",
        "restore_layer": "Restore layer",
        "n_source_features": "# source feats",
        "n_restore_features": "# restore feats",

        "mean_m_mean_drop_after_x_ablation": "X→M act. drop",
        "rate_x_ablation_decreases_m_activation": "Rate X lowers M",

        "mean_x_to_y_logit_loss": "X→Y logit loss",
        "rate_x_ablation_lowers_logit": "Rate X lowers Y",

        "mean_m_to_y_logit_loss": "M→Y logit loss",
        "rate_m_ablation_lowers_logit": "Rate M lowers Y",

        "mean_mediation_restored_delta": "Restore Δlogit",
        "mean_mediation_recovery_ratio": "Recovery ratio",
        "rate_restore_increases_logit_after_x_ablation": "Rate restore helps",

        "rate_supports_route_instance": "Route support rate",
        "supports_route_pair": "Supports route",

        "flip_rate_clean_to_x_ablate": "Flip: clean→X ablate",
        "flip_rate_clean_to_m_ablate": "Flip: clean→M ablate",
        "flip_rate_x_ablate_to_restore": "Flip: X ablate→restore",
    }

    table = table.rename(columns=rename_map)

    # Round numeric columns.
    for col in table.columns:
        if pd.api.types.is_numeric_dtype(table[col]):
            if "layer" in col.lower() or "# " in col:
                table[col] = table[col].astype(int)
            else:
                table[col] = table[col].apply(lambda x: _fmt_float(x, 3))

    # Save.
    out_csv = os.path.join(output_dir, "adjacent_group_pair_results_table.csv")
    out_latex = os.path.join(output_dir, "adjacent_group_pair_results_table.tex")

    table.to_csv(out_csv, index=False)

    latex = table.to_latex(
        index=False,
        escape=True,
        longtable=False,
        caption="Adjacent-layer group path-patching results by emotion and layer pair.",
        label="tab:adjacent_group_path_patching",
    )
    with open(out_latex, "w") as f:
        f.write(latex)

    print("Saved pair table:", out_csv)
    print("Saved LaTeX table:", out_latex)

    display(table)
    return table


def make_emotion_result_table(
    emotion_summary_path: str,
    output_dir: str,
):
    df = pd.read_csv(emotion_summary_path)

    keep_cols = [
        "target_emotion",
        "n_layer_pairs",
        "supported_pair_rate",

        "mean_x_to_m_activation_drop",
        "mean_rate_x_ablation_decreases_m_activation",

        "mean_x_to_y_logit_loss",
        "mean_rate_x_ablation_lowers_logit",

        "mean_m_to_y_logit_loss",
        "mean_rate_m_ablation_lowers_logit",

        "mean_mediation_restored_delta",
        "mean_mediation_recovery_ratio",
        "max_mediation_recovery_ratio",
        "mean_rate_restore_increases_logit_after_x_ablation",

        "mean_rate_supports_route_instance",

        "mean_clean_to_x_ablate_flip",
        "mean_clean_to_m_ablate_flip",
        "mean_x_ablate_to_restore_flip",
    ]

    keep_cols = [c for c in keep_cols if c in df.columns]
    table = df[keep_cols].copy()

    rename_map = {
        "target_emotion": "Emotion",
        "n_layer_pairs": "# layer pairs",
        "supported_pair_rate": "Supported pair rate",

        "mean_x_to_m_activation_drop": "Mean X→M act. drop",
        "mean_rate_x_ablation_decreases_m_activation": "Rate X lowers M",

        "mean_x_to_y_logit_loss": "Mean X→Y logit loss",
        "mean_rate_x_ablation_lowers_logit": "Rate X lowers Y",

        "mean_m_to_y_logit_loss": "Mean M→Y logit loss",
        "mean_rate_m_ablation_lowers_logit": "Rate M lowers Y",

        "mean_mediation_restored_delta": "Mean restore Δlogit",
        "mean_mediation_recovery_ratio": "Mean recovery ratio",
        "max_mediation_recovery_ratio": "Max recovery ratio",
        "mean_rate_restore_increases_logit_after_x_ablation": "Rate restore helps",

        "mean_rate_supports_route_instance": "Route support rate",

        "mean_clean_to_x_ablate_flip": "Flip: clean→X ablate",
        "mean_clean_to_m_ablate_flip": "Flip: clean→M ablate",
        "mean_x_ablate_to_restore_flip": "Flip: X ablate→restore",
    }

    table = table.rename(columns=rename_map)

    for col in table.columns:
        if pd.api.types.is_numeric_dtype(table[col]):
            if "# " in col:
                table[col] = table[col].astype(int)
            else:
                table[col] = table[col].apply(lambda x: _fmt_float(x, 3))

    out_csv = os.path.join(output_dir, "adjacent_group_emotion_results_table.csv")
    out_latex = os.path.join(output_dir, "adjacent_group_emotion_results_table.tex")

    table.to_csv(out_csv, index=False)

    latex = table.to_latex(
        index=False,
        escape=True,
        longtable=False,
        caption="Emotion-level adjacent-layer group path-patching results.",
        label="tab:adjacent_group_emotion_summary",
    )
    with open(out_latex, "w") as f:
        f.write(latex)

    print("Saved emotion table:", out_csv)
    print("Saved LaTeX table:", out_latex)

    display(table)
    return table


# ============================================================
# Run
# ============================================================

pair_table = make_pair_result_table(
    pair_summary_path=PAIR_SUMMARY_PATH,
    output_dir=TABLE_OUT_DIR,
    top_k_per_emotion=None,  # set to 3 if you only want top 3 layer pairs per emotion
)

emotion_table = make_emotion_result_table(
    emotion_summary_path=EMOTION_SUMMARY_PATH,
    output_dir=TABLE_OUT_DIR,
)

In [ ]:
import os
import pandas as pd
from IPython.display import display

# ============================================================
# Config
# ============================================================

MODELS = [
    "google/gemma-2-2b",
    "google/gemma-2-9b",
    "google/gemma-3-4b-pt",
    "google/gemma-3-4b-it",
    "meta-llama/Llama-3.1-8B",
    "google/gemma-3-12b-pt",
    "google/gemma-3-12b-it",
]

BASE_DATA_DIR = "data"

TABLE_OUT_DIR = os.path.join(
    BASE_DATA_DIR,
    "all_models_adjacent_group_route_mediation",
    "tables",
)
os.makedirs(TABLE_OUT_DIR, exist_ok=True)


# ============================================================
# Helpers
# ============================================================

def _fmt_float(x, digits=3):
    if pd.isna(x):
        return ""
    return round(float(x), digits)


def get_feature_dir(model_name: str, sae_size: str | None = None):
    model_slug = model_name.replace("/", "-")

    if sae_size is None:
        return os.path.join(BASE_DATA_DIR, f"{model_slug}-feature-act")

    return os.path.join(BASE_DATA_DIR, f"{model_slug}-feature-act-{sae_size}")


# ============================================================
# Main
# ============================================================

def make_all_model_emotion_table(
    models,
    table_out_dir,
    sae_size_by_model=None,
):
    sae_size_by_model = sae_size_by_model or {}

    all_tables = []

    for model_name in models:
        sae_size = sae_size_by_model.get(model_name, None)

        feature_dir = get_feature_dir(
            model_name,
            sae_size=sae_size,
        )

        emotion_summary_path = os.path.join(
            feature_dir,
            "adjacent_group_route_mediation",
            "adjacent_group_path_patching",
            "adjacent_group_emotion_summary.csv",
        )

        if not os.path.exists(emotion_summary_path):
            print(f"[Skip] Missing file for {model_name}:")
            print(emotion_summary_path)
            continue

        df = pd.read_csv(emotion_summary_path)

        keep_cols = [
            "target_emotion",

            # X -> M
            "mean_x_to_m_activation_drop",

            # X -> Y
            "mean_x_to_y_logit_loss",

            # M -> Y
            "mean_m_to_y_logit_loss",

            # X -> M -> Y
            "mean_mediation_restored_delta",
            "mean_mediation_recovery_ratio",
        ]

        keep_cols = [c for c in keep_cols if c in df.columns]

        table = df[keep_cols].copy()

        table.insert(0, "model", model_name)
        table.insert(1, "sae_size", sae_size if sae_size is not None else "")

        all_tables.append(table)

    if len(all_tables) == 0:
        raise FileNotFoundError("No emotion summary files found.")

    # ============================================================
    # Combine all model tables
    # ============================================================

    all_df = pd.concat(all_tables, ignore_index=True)

    # ============================================================
    # Add average row PER EMOTION across models
    # ============================================================

    numeric_cols = [
        c for c in all_df.columns
        if pd.api.types.is_numeric_dtype(all_df[c])
    ]

    avg_df = (
        all_df
        .groupby("target_emotion", dropna=False)[numeric_cols]
        .mean()
        .reset_index()
    )

    avg_df["model"] = "ALL_MODELS_AVG"
    avg_df["sae_size"] = ""

    # Match column order
    avg_df = avg_df[all_df.columns]

    # Append
    all_df = pd.concat([all_df, avg_df], ignore_index=True)

    # ============================================================
    # Rename columns
    # ============================================================

    rename_map = {
        "model": "Model",
        "sae_size": "SAE size",
        "target_emotion": "Emotion",
        "mean_x_to_m_activation_drop": "Mean X→M act. drop",
        "mean_x_to_y_logit_loss": "Mean X→Y logit loss",
        "mean_m_to_y_logit_loss": "Mean M→Y logit loss",
        "mean_mediation_restored_delta": "Mean restore Δlogit",
        "mean_mediation_recovery_ratio": "Mean recovery rate",
    }

    all_df = all_df.rename(columns=rename_map)

    # ============================================================
    # Format numeric columns
    # ============================================================

    for col in all_df.columns:
        if pd.api.types.is_numeric_dtype(all_df[col]):
            all_df[col] = all_df[col].apply(
                lambda x: _fmt_float(x, 3)
            )

    # ============================================================
    # Emotion order
    # ============================================================

    emotion_order = [
        "anger",
        "disgust",
        "fear",
        "joy",
        "sadness",
        "surprise",
    ]

    all_df["Emotion"] = pd.Categorical(
        all_df["Emotion"],
        categories=emotion_order,
        ordered=True,
    )

    # ============================================================
    # Sort
    # ============================================================

    all_df = all_df.sort_values(
        ["Model", "SAE size", "Emotion"]
    ).reset_index(drop=True)

    # ============================================================
    # Save
    # ============================================================

    out_csv = os.path.join(
        table_out_dir,
        "all_models_adjacent_group_emotion_mean_results.csv",
    )

    out_latex = os.path.join(
        table_out_dir,
        "all_models_adjacent_group_emotion_mean_results.tex",
    )

    all_df.to_csv(out_csv, index=False)

    latex = all_df.to_latex(
        index=False,
        escape=True,
        longtable=False,
        caption="Mean adjacent-layer group path-patching results per emotion across models.",
        label="tab:all_models_adjacent_group_emotion_mean",
    )

    with open(out_latex, "w") as f:
        f.write(latex)

    print("Saved CSV:", out_csv)
    print("Saved LaTeX:", out_latex)

    display(all_df)

    return all_df


# ============================================================
# Run
# ============================================================

SAE_SIZE_BY_MODEL = {
    "google/gemma-3-4b-pt": "big",
    "google/gemma-3-4b-it": "big",
    "google/gemma-3-12b-pt": "small",
    "google/gemma-3-12b-it": "small",
}

all_model_emotion_table = make_all_model_emotion_table(
    models=MODELS,
    table_out_dir=TABLE_OUT_DIR,
    sae_size_by_model=SAE_SIZE_BY_MODEL,
)